In [ ]:
import os
import pprint
import math
from glob import glob
import itertools
import pandas as pd
import numpy as np
import yaml
import json
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import clear_output

from pyomo.environ import (
    ConcreteModel,
    Var,
    Param,
    Expression,
    Constraint,
    value,
    Reference,
    assert_optimal_termination,
    check_optimal_termination,
    units as pyunits,
)
from pyomo.opt.results import SolverStatus, TerminationCondition
from idaes.core.util.model_statistics import (
    degrees_of_freedom,
    number_unfixed_variables,
)
from idaes.core import FlowsheetBlock, UnitModelCostingBlock
import idaes.core.util.scaling as iscale

from watertap.core.wt_database import Database
from watertap.core.zero_order_properties import WaterParameterBlock
import watertap.unit_models.zero_order as zo
from watertap.costing import ZeroOrderCosting
from watertap.core.solvers import get_solver

# from watertap.kurby import *

solver = get_solver()

rho = 1000 * pyunits.kg / pyunits.m**3  # density of water

default_comps = [
    "toc",
    "tkn",
    "tss",
    "cod",
    "tds",
    "nitrogen",
    "phosphates",
    "phosphorus",
    "struvite",
    "nonbiodegradable_cod",
    "hydrogen",
    "ammonium_as_nitrogen",
    "nitrate",
    "bod",
    "organic_solid",
    "organic_liquid",
    "iron",
    "filtration_media",
    "peracetic_acid",
    "total_coliforms_fecal_ecoli",
]

zo_units = pd.read_csv("zo_unit_summary.csv", index_col=0)
zo_units["unit_model_class"] = zo_units.index
# zo_units.reset_index(inplace=True)
zo_units


In [ ]:


def build_results_dict(
    b,
    components=[Var, Expression, Param],
    skips=[],
    descend_into=True,
):
    """
    Function to build a results dictionary with all components on block
    components: Pyomo objects to include in results_dict
    skips: component names (or parts of names) to skip

    returns: dictionary with desired model component results to store

    To be used with results_dict_append function
    """
    if skips == []:
        skips = skips_default
    results_dict = {}

    for c in components:
        for v in b.component_objects(c, descend_into=descend_into):
            if any(s in v.name for s in skips):
                continue
            elif v.is_indexed():
                for i, vv in v.items():
                    results_dict[vv.name] = []
            else:
                results_dict[v.name] = []

    return results_dict


def results_dict_append(
    b,
    results_dict,
    components=[Var, Expression, Param],
    tmp_results_dict=None,
):
    """
    Add to results_dict after model solve

    results_dict: dictionary meant to store model results
    components: model components included in dictionary
    tmp_results_dict: temporary results dictionary for nested loop meant to add to larger results dictionary

    returns: the same results_dict, but with another "row" added

    """

    appended = list()

    if tmp_results_dict is None:
        for c in components:
            for v in b.component_objects(c):
                if v.is_indexed():
                    # print(v.name)
                    # idx = [*v._index_set]
                    for i, vv in v.items():
                        if vv.name in results_dict.keys():
                            if vv.name in appended:
                                continue
                            try:
                                results_dict[vv.name].append(value(vv))
                                appended.append(vv.name)
                            except:
                                pass
                else:
                    if v.name in results_dict.keys():
                        if v.name in appended:
                            continue
                        try:
                            results_dict[v.name].append(value(v))
                            appended.append(v.name)
                        except:
                            pass

    else:
        for k, v in tmp_results_dict.items():
            if k in appended or k not in results_dict.keys():
                continue
            results_dict[k].append(v)
            appended.append(k)

    return results_dict


In [ ]:
def find_key(d, target_key):
    if isinstance(d, dict):
        for key, value in d.items():
            if key == target_key:
                return value
            result = find_key(value, target_key)
            if result is not None:
                return result
    elif isinstance(d, list):
        # Handle case where values are lists of dicts
        for item in d:
            result = find_key(item, target_key)
            if result is not None:
                return result
    return None


def parse_units(s):

    if "/" in s:
        num, denom = s.split("/")
        num = num.strip()
        num = getattr(pyunits, num)
        denom = denom.strip()
        denom = getattr(pyunits, denom)
        return num / denom


def build_zo_model(
    unit_class,
    comps=None,
    subtype="default",
    flow_mgd=0.1,
    conc=100,
    unit_config=dict(),
    costing_method_arguments=dict(),
):

    m = ConcreteModel()
    m.db = Database()
    m.fs = FlowsheetBlock(dynamic=False)
    m.fs.costing = ZeroOrderCosting()
    # m.fs.costing.base_currency = pyunits.USD_2008
    m.fs.costing.base_currency = pyunits.USD_2023

    if comps is None:
        comps = default_comps

    flows_to_register = [
        "catalyst_ATHTL",
        "polymer",
        "magnesium_chloride",
        "catalyst_HTG",
        "hydrogen_product",
        "filtration_media",
        "filtration_media_disposal",
        "disinfection_solution",
        "struvite_product",
    ]
    for flow in flows_to_register:
        m.fs.costing.register_flow_type(flow, 0)

    m.fs.properties = WaterParameterBlock(
        solute_list=comps,
    )
    unit = getattr(zo, unit_class)

    m.fs.unit = unit(
        property_package=m.fs.properties,
        database=m.db,
        process_subtype=subtype,
        **unit_config,
    )
    m.fs.unit.costing = UnitModelCostingBlock(
        flowsheet_costing_block=m.fs.costing,
        **costing_method_arguments,
    )

    flow_mgd = flow_mgd * pyunits.Mgallons / pyunits.day
    conc = conc * pyunits.mg / pyunits.L

    flow_mass_water = flow_mgd * rho

    comp_flow_dict = dict()

    for c in comps:
        # conc is the same for all comps
        comp_flow_dict[c] = flow_mgd * conc

    prop_in = (
        m.fs.unit.properties_in[0]
        if hasattr(m.fs.unit, "properties_in")
        else m.fs.unit.properties[0]
    )
    prop_out = (
        m.fs.unit.properties_out[0] if hasattr(m.fs.unit, "properties_out") else prop_in
    )

    prop_in.flow_mass_comp["H2O"].fix(flow_mass_water)
    m.fs.properties.set_default_scaling(
        "flow_mass_comp", 1 / value(flow_mass_water), index="H2O"
    )
    for c in comps:
        prop_in.flow_mass_comp[c].fix(comp_flow_dict[c])
        m.fs.properties.set_default_scaling(
            "flow_mass_comp", 1 / value(comp_flow_dict[c]), index=(c)
        )

    data = m.db.get_unit_operation_parameters(m.fs.unit._tech_type, subtype=subtype)

    m.fs.unit.load_parameters_from_database(use_default_removal=True)
    if m.fs.unit.find_component("removal_frac_mass_comp") is not None:
        for c in comps:
            if c in data["removal_frac_mass_comp"].keys():
                m.fs.unit.removal_frac_mass_comp[0, c].fix(
                    data["removal_frac_mass_comp"][c]["value"]
                )
            else:
                # default removal_fracs of 0 can cause solving problems
                m.fs.unit.removal_frac_mass_comp[0, c].fix(0.1)

    m.fs.costing.cost_process()
    m.fs.costing.add_LCOW(prop_out.flow_vol)
    try:
        m.fs.costing.add_specific_energy_consumption(prop_out.flow_vol, name="SEC")
    except:
        print(f"\nNo energy consumption for {unit_class} with subtype {subtype}\n")

    iscale.calculate_scaling_factors(m)
    return m

In [ ]:
bounds_units = [
    "Mgallons/day",
    "acre",
    "ft3",
    "ft^2",
    "gal/day",
    "gallons",
    "liter/second",
    "m^3",
    "m^3/day",
    "mg/L",
]
flow_bounds_units = [
    "Mgallons/day",
    "gal/day",
    "liter/second",
    "m^3/day",
]

def run_zo_flow_sweep(
    u=None,
    comps=None,
    subtype="default",
    flow_mgd=0.1,
    # flow_mgd_bou=0.1,
    conc=100,
    x=None,
    y=None,
    n_flow_pts=10,
):

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)

    u_info = zo_units.loc[u]
    # print(u_info)
    # Determine validity range
    if u_info.validity_range is not np.nan:
        vr = json.loads(u_info.validity_range.replace("'", '"'))
        subtypes = json.loads(u_info.subtypes.replace("'", '"'))
        # print(vr)
        if vr["units"] in flow_bounds_units:
            # print(u)
            pu = parse_units(vr["units"])
            lb = value(
                pyunits.convert(
                    vr["lower_bound"] * pu, to_units=pyunits.Mgallons / pyunits.day
                )
            )
            ub = value(
                pyunits.convert(
                    vr["upper_bound"] * pu, to_units=pyunits.Mgallons / pyunits.day
                )
            )
            if lb > 1:
                lb = 0.1
            elif lb == 0:
                lb = 0.1  # set a small positive value to avoid zero
                assert lb < ub
            flow_sweep = np.linspace(0.1, ub, n_flow_pts)
        else:
            flow_sweep = np.linspace(0.1, 20, n_flow_pts)
    else:
        vr = None
        subtypes = ["default"]
        flow_sweep = np.linspace(0.1, 20, n_flow_pts)

    rd_df = pd.DataFrame()
    for subtype in subtypes:
        m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=1, conc=conc)
        rd = build_results_dict(m, skips=["fs.unit.properties"])
        rd["flow_mgd"] = list()
        rd["subtype"] = list()
        print(f"\n\nRUNNING {u} with subtype {subtype}\n\n")
        for flow in flow_sweep:
            print(f"\n\tflow = {flow} Mgallons/day\n")
            m = build_zo_model(
                u, comps=comps, subtype=subtype, flow_mgd=flow, conc=conc
            )
            m.fs.unit.initialize()
            # try:
            results = solver.solve(m, tee=False)
            assert_optimal_termination(results)
            # except:
            #     continue
            rd = results_dict_append(m, rd)
            rd["flow_mgd"].append(flow)
            rd["subtype"].append(subtype)
        tmp = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
        tmp["unit_class"] = u
        tmp["unit"] = zo_units.loc[u, "tech_type"]
        if vr is not None:
            tmp["validity_range_units"] = vr["units"]
            tmp["validity_range_lb"] = vr["lower_bound"]
            tmp["validity_range_ub"] = vr["upper_bound"]
        rd_df = pd.concat([rd_df, tmp])
    # print(subtypes)
    # rd = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
    clear_output(wait=True)
    # for k, v in rd.items():
    #     print(f"{k}: {len(v)}")
    # return m, rd_df
    return rd_df

In [ ]:
pretty_subtypes = {
    "default": "Default",
    "anion_exchange": "Anion Exchange",
    "cation_exchange": "Cation Exchange",
    "alum": "Alum",
    "ammonia": "Ammonia",
    "anhydrous_ammonia": "Anhydrous Ammonia",
    "anti-scalant": "Anti-Scalant",
    "caustic_soda": "Caustic Soda",
    "ferric_chloride": "Ferric Chloride",
    "hydrochloric_acid": "Hydrochloric Acid",
    "lime": "Lime",
    "sodium_bisulfite": "Sodium Bisulfite",
    "sulfuric_acid": "Sulfuric Acid",
    "chlorine": "Chlorine",
}


def plot_zo_results(
    rd,
    fig=None,
    axes=None,
    x_col="flow_mgd",
    y_cols=None,
    hue_col=None,
    leg_idx=(1, 0),  # which of the plots get the legend
    set_kwargs=dict(),
    plot_kwargs=dict(),
    colormap="tab20",
    skip_default=False,
):
    def _dollar_formatter(x, _pos):
        if x >= 1e6:
            return f"${x / 1e6:.1f}M"
        elif x >= 1e3:
            return f"${x / 1e3:.0f}k"
        else:
            return f"${x:.0f}"

    def _LCOW_formatter(x, _pos):
        if x < 1e-2:
            return f"{x:.0e}"
        elif 1e-2 <= x < 1:
            return f"{x:.2f}"
        else:
            return f"{x:.1f}"

    def plot_the_row(row):
        print(row, y_cols)
        y = y_cols[row]
        r = rd.sort_values(by=y)
        if hue_col is None:
            ax_lin.plot(r[x_col], r[y], linewidth=1.8, label="Default")
            ax_log.loglog(
                r[x_col],
                r[y],
                linewidth=1.8,
                # label=pretty_subtypes.get(hue, hue),
            )
        else:
            hues = rd[hue_col].unique()
            color_list = [cmap(i) for i in np.linspace(0, 1, 20)]

            for i, hue in enumerate(hues):
                if hue == "default" and skip_default:
                    # sometimes default is the same as another subtype
                    continue
                color = color_list[i]
                subset = rd[rd[hue_col] == hue].copy()
                subset = subset.sort_values(by=x_col)
                h = hue.replace("_", " ").title() if isinstance(hue, str) else hue
                ax_lin.plot(
                    subset[x_col],
                    subset[y],
                    linewidth=1.8,
                    label=pretty_subtypes.get(hue, h),
                    color=color,
                )
                ax_log.loglog(
                    subset[x_col],
                    subset[y],
                    linewidth=1.8,
                    label=pretty_subtypes.get(hue, h),
                    color=color,
                )

    if y_cols is None:
        y_cols = ("fs.costing.LCOW", "fs.costing.total_capital_cost")
    if (fig, axes) == (None, None):
        fig, axes = plt.subplots(
            nrows=len(y_cols), ncols=2, figsize=(8, 4.25 * len(y_cols))
        )

    if colormap == "tab10":
        raise ValueError(f"Don't use that colormap {colormap}.")
    cmap = getattr(plt.cm, colormap)
    color_list = [cmap(i) for i in np.linspace(0, 1, 20)]
    # print(color_list)

    for row, ax in enumerate(axes):
        if row + 1 > len(y_cols):
            break
        ax_lin, ax_log = [*ax]
        plot_the_row(row)
        for col, a in enumerate(ax):
            a.set(**set_kwargs)
            a.grid(True, alpha=0.3, which="both")
            if row == 0:
                a.yaxis.set_major_formatter(plt.FuncFormatter(_LCOW_formatter))
            if row == 1:
                a.yaxis.set_major_formatter(plt.FuncFormatter(_dollar_formatter))

    axes[*leg_idx].legend(fontsize=9, loc="upper left", framealpha=0.9)
    fig.tight_layout()
    fig.suptitle(rd.unit_class.unique()[0], y=1.05)
    return fig, axes


# rd = pd.read_csv("results/uv_flow_sec_sweep_results.csv")
# fig, axes = plot_zo_results(rd, hue_col="sec", skip_default=False)

# rd = pd.read_csv("results/ALL_zo_flow_sweep_results_all2.csv")
# # # rd.reset_index(inplace=True)
# # pu = "mbr"
# # # for pu in rd.unit.unique():
# # df = rd[rd.unit == pu].copy()
# # # df.set_index(["flow_mgd", "subtype"], inplace=True)
# # fig, axes = plot_zo_results(df)
# # fig.savefig(f"figs/{pu}_flow_sweep_plots.png", dpi=300)

# pu = "screen"
# # for pu in rd.unit.unique():
# df = rd[rd.unit == pu].copy()
# # df.set_index(["flow_mgd", "subtype"], inplace=True)
# fig, axes = plot_zo_results(df, hue_col="subtype")
# fig.savefig(f"figs/{pu}_flow_sweep_plots.png", dpi=300)
# # break

# RUN ZO FLOW SWEEPS

In [ ]:
# will be run with custom function or are just not run 


unit_skips = [
    "anaerobic_mbr_mec",
    "autothermal_hydrothermal_liquefaction",
    # "bioreactor",
    "brine_concentrator",
    "CANDO_P",
    "centrifuge",
    "chlorination",
    "chemical_addition",
    "cloth_media_filtration",
    "cofermentation",
    "constructed_wetlands",
    "coag_and_floc",
    "crystallizer",
    "deep_well_injection",
    "dmbr",
    "electrochemical_nutrient_removal",
    "electrocoagulation",
    "evaporation_pond",
    "gac",
    "gas_sparged_membrane",
    "hrcs",
    "hydrothermal_gasification",
    "hydrothermal_liquefaction",
    "iron_and_manganese_removal",
    "mabr",
    "magprex",
    "filter_press",
    "ion_exchange",
    "membrane_evaporator",
    "metab",
    "microbial_battery",
    # "municipal_wwtp",
    "ozone",
    "ozone_aop",
    "peracetic_acid_disinfection",
    "photothermal_membrane",
    "pump_electricity",
    "sedimentation",
    # "secondary_treatment_wwtp",
    "storage_tank",
    "struvite_classifier",
    "suboxic_anaerobic_sludge_process",
    # "smp",
    "suboxic_activated_sludge_process",
    "supercritical_salt_precipitation",
    "uv",
    "uv_aop",
    "vfa_recovery",
    "well_field",
]
didnt_work = ["feed", "microscreen_filtration"]
need_special_comps = ["anaerobic_digestion_reactive", "CANDOP"]
n_flow_pts = 25
comps = ["tds"]
zo_res = pd.DataFrame()
for unit_model_class, row in zo_units.iterrows():
    if row["tech_type"] in unit_skips + didnt_work + need_special_comps:
        print(f"Skipping {row['tech_type']}")
        continue

    print(f"\nRunning {row['tech_type']}...\n\n")
    rd = run_zo_flow_sweep(u=unit_model_class, n_flow_pts=n_flow_pts, comps=comps)

    rd.to_csv(f"results/{row['tech_type']}_zo_flow_sweep_results.csv", index=False)
    zo_res = pd.concat([zo_res, rd])
    break

#     zo_res.to_csv("results/ALL_zo_flow_sweep_results_all-tmp.csv", index=False)
# zo_res.to_csv("results/ALL_zo_flow_sweep_results_all.csv", index=False)

# RUN OZONE SWEEP

In [ ]:
def run_ozone_sweep():
    u = "OzoneZO"
    comps = ["toc"]
    subtype = "default"
    flow_mgd = 1
    conc = 1
    n_flow_pts = 25
    n_dose_pts = 10

    flow_sweep = np.linspace(1, 25, n_flow_pts)
    dose_sweep = np.linspace(1, 25, n_dose_pts)
    # doses = [1, 5, 10, 15, 20, 25]

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    # return m
    ozone_res = pd.DataFrame()

    for flow in flow_sweep:
        rd = build_results_dict(m, skips=["fs.unit.properties"])
        rd["flow_mgd"] = list()
        rd["dose"] = list()
        for dose in dose_sweep:
            print(f"Running flow = {flow} MGD, dose = {dose} mg/L")
            m = build_zo_model(
                u, comps=comps, subtype=subtype, flow_mgd=flow, conc=0.01
            )
            # CAPEX is function of ozone_consumption, which is a function of TOC.
            # unfix concentration_time so we don't have to worry about TOC :)
            m.fs.unit.ozone_consumption.fix(dose * pyunits.mg / pyunits.L)
            m.fs.unit.concentration_time.unfix()
            m.fs.unit.concentration_time.setlb(0)
            m.fs.unit.concentration_time.setub(None)
            m.fs.unit.initialize()

            results = solver.solve(m, tee=False)
            assert_optimal_termination(results)
            rd = results_dict_append(m, rd)
            rd["flow_mgd"].append(flow)
            rd["dose"].append(dose)
            # break
        tmp = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
        tmp["subtype"] = subtype
        tmp["unit_class"] = u
        tmp["unit"] = zo_units.loc[u, "tech_type"]
        # tmp["validity_range_units"] = "Mgallons/day"
        # tmp["validity_range_lb"] = 1
        # tmp["validity_range_ub"] = 25
        ozone_res = pd.concat([ozone_res, tmp])
    return ozone_res

ozone_res = run_ozone_sweep()
ozone_res.to_csv("results/ozone_flow_dose_sweep_results.csv", index=False)

# RUN OZONE AOP SWEEP

In [ ]:
def run_ozone_aop_sweep():
    u = "OzoneAOPZO"
    comps = ["toc"]
    subtype = "default"
    flow_mgd = 1
    conc = 1
    n_flow_pts = 10

    flow_sweep = np.linspace(1, 25, n_flow_pts)
    dose_sweep = np.linspace(1, 25, n_flow_pts)
    # doses = [1, 5, 10, 15, 20, 25]

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    ozone_aop_res = pd.DataFrame()

    for flow in flow_sweep:
        rd = build_results_dict(m, skips=["fs.unit.properties"])
        rd["flow_mgd"] = list()
        rd["dose"] = list()
        rd["oxidant_dose"] = list()
        for dose in dose_sweep:
            for oxidant_dose in [1, 10]:
                print(
                    f"Running flow = {flow} MGD, dose = {dose} mg/L, oxidant_dose = {oxidant_dose} mg/L"
                )
                m = build_zo_model(
                    u, comps=comps, subtype=subtype, flow_mgd=flow, conc=0.01
                )

                m.fs.unit.ozone_consumption.fix(dose * pyunits.mg / pyunits.L)
                m.fs.unit.oxidant_dose.fix(oxidant_dose * pyunits.mg / pyunits.L)
                m.fs.unit.oxidant_ozone_ratio.unfix()
                m.fs.unit.concentration_time.unfix()
                m.fs.unit.concentration_time.setlb(0)
                m.fs.unit.concentration_time.setub(None)
                print(f"dof = {degrees_of_freedom(m)}")
                m.fs.unit.initialize()

                results = solver.solve(m, tee=False)
                assert_optimal_termination(results)
                rd = results_dict_append(m, rd)
                rd["flow_mgd"].append(flow)
                rd["dose"].append(dose)
                rd["oxidant_dose"].append(oxidant_dose)
            clear_output(wait=True)
        tmp = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
        tmp["subtype"] = subtype
        tmp["unit_class"] = u
        tmp["unit"] = zo_units.loc[u, "tech_type"]
        # tmp["validity_range_units"] = "Mgallons/day"
        # tmp["validity_range_lb"] = 1
        # tmp["validity_range_ub"] = 25
        ozone_aop_res = pd.concat([ozone_aop_res, tmp])

    return ozone_aop_res

ozone_aop_res = run_ozone_aop_sweep()
ozone_aop_res.to_csv("results/ozone_aop_flow_dose_sweep_results.csv", index=False)

# RUN CHEM ADDITION SWEEP

In [ ]:
def run_chemical_addition_sweep():
    u = "ChemicalAdditionZO"
    comps = ["tds"]
    subtype = "ammonia"
    flow_mgd = 1
    conc = 1
    n_flow_pts = 10
    n_dose_pts = 25

    flow_sweep = np.linspace(1, 25, n_flow_pts)
    dose_sweep = [1, 5, 10, 20, 25, 50, 75, 100, 125, 250, 500]

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    subtypes = [
        "alum",
        "ammonia",
        "anhydrous_ammonia",
        "anti-scalant",
        "caustic_soda",
        "ferric_chloride",
        "hydrochloric_acid",
        "lime",
        "sodium_bisulfite",
        "sulfuric_acid",
        "chlorine",
    ]

    chem_add_res = pd.DataFrame()
    for subtype in subtypes:
        rd = build_results_dict(m, skips=["fs.unit.properties"])
        rd["flow_mgd"] = list()
        rd["dose"] = list()
        for flow in flow_sweep:
            # rd = build_results_dict(m, skips=["fs.unit.properties"])
            # rd["flow_mgd"] = list()
            # rd["dose"] = list()
            for dose in dose_sweep:
                print(
                    f"Running {subtype.upper()},flow = {flow} MGD, dose = {dose} mg/L"
                )
                m = build_zo_model(
                    u, comps=comps, subtype=subtype, flow_mgd=flow, conc=conc
                )
                # m.fs.unit.initialize()
                # m.fs.costing.initialize()

                m.fs.unit.chemical_dosage.fix(dose * pyunits.mg / pyunits.L)
                m.fs.unit.initialize()
                results = solver.solve(m, tee=False)
                assert_optimal_termination(results)
                rd = results_dict_append(m, rd)
                rd["flow_mgd"].append(flow)
                rd["dose"].append(dose)
        clear_output(wait=True)
        tmp = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
        tmp["subtype"] = subtype
        tmp["unit_class"] = u
        tmp["unit"] = zo_units.loc[u, "tech_type"]
        # tmp["validity_range_units"] = "Mgallons/day"
        # tmp["validity_range_lb"] = 1
        # tmp["validity_range_ub"] = 25
        chem_add_res = pd.concat([chem_add_res, tmp])
    return chem_add_res


chem_add_res = run_chemical_addition_sweep()
chem_add_res.to_csv(
    "results/chemical_addition_flow_dose_sweep_results.csv", index=False
)
# rd = pd.read_csv("results/chemical_addition_flow_dose_sweep_results.csv")
# rd = rd[rd.dose == 1].copy()
# fig, axes = plot_zo_results(rd)
# rd = pd.read_csv("results/chemical_addition_flow_dose_sweep_results.csv")
# rd = rd[rd.dose == 100].copy()
# fig, axes = plot_zo_results(rd)

# RUN GAC SWEEP

In [ ]:
def run_gac_sweep():
    u = "GACZO"
    comps = ["tds"]
    subtype = "gravity"
    flow_mgd = 1
    conc = 100
    n_flow_pts = 20

    flow_sweep = np.linspace(1, 25, n_flow_pts)
    ebct_sweep = [7.5, 15]

    cma = dict(
        number_of_parallel_units=1
    )
    cma = dict(costing_method_arguments=cma)
    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc, costing_method_arguments=cma)
    m.fs.unit.initialize()
    m.fs.costing.initialize()
    subtypes = ["pressure_vessel", "gravity"]

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    gac_res = pd.DataFrame()
    # rd["ebct"] = list()
    for subtype in subtypes:
        rd = build_results_dict(m, skips=["fs.unit.properties"])
        rd["flow_mgd"] = list()
        for flow in flow_sweep:
            # for ebct in ebct_sweep:
            print(f"Running {subtype.upper()} for flow = {flow} MGD")
            m = build_zo_model(
                u, comps=comps, subtype=subtype, flow_mgd=flow, conc=conc
            )
            # m.fs.unit.empty_bed_contact_time.fix(ebct * pyunits.minute)
            m.fs.unit.initialize()
            results = solver.solve(m, tee=False)
            assert_optimal_termination(results)
            rd = results_dict_append(m, rd)
            rd["flow_mgd"].append(flow)
            print(f"LCOW = {value(m.fs.costing.LCOW):.2f} $/m^3")
        tmp = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
        tmp["subtype"] = subtype
        tmp["unit_class"] = u
        tmp["unit"] = zo_units.loc[u, "tech_type"]
        # tmp["validity_range_units"] = "Mgallons/day"
        # tmp["validity_range_lb"] = 1
        # tmp["validity_range_ub"] = 25
        
        gac_res = pd.concat([gac_res, tmp])
        print(len(gac_res))
        clear_output(wait=True)
    return gac_res


gac_res = run_gac_sweep()
gac_res.to_csv("results/gac_flow_ebct_sweep_results.csv", index=False)


# RUN IX SWEEP

In [ ]:
def run_ion_exchange_sweep():
    u = "IonExchangeZO"
    comps = ["tds"]
    subtype = "default"
    flow_mgd = 1
    conc = 100
    n_flow_pts = 20
    n_tds_pts = 20

    flow_sweep = np.linspace(1, 25, n_flow_pts)
    tds_sweep = {
        "anion_exchange": np.linspace(
            50, 150, n_tds_pts
        ),  # mg/L, model uses TDS for sulfate
        "cation_exchange": np.linspace(
            200, 1000, n_tds_pts
        ),  # mg/L, model uses TDS for hardness
    }

    subtypes = [
        "anion_exchange",
        "cation_exchange",
    ]

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    ion_ex_res = pd.DataFrame()
    for subtype in subtypes:
        rd = build_results_dict(m)
        rd["flow_mgd"] = list()
        rd["tds"] = list()
        for flow in flow_sweep:
            for tds in tds_sweep[subtype]:
                print(
                    f"\nRunning {subtype.upper()} with flow = {flow} MGD, tds = {tds} mg/L\n\n"
                )
                m = build_zo_model(
                    u, comps=comps, subtype=subtype, flow_mgd=flow, conc=tds
                )

                m.fs.unit.initialize()

                results = solver.solve(m, tee=False)
                assert_optimal_termination(results)
                rd = results_dict_append(m, rd)
                rd["flow_mgd"].append(flow)
                rd["tds"].append(tds)

        tmp = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
        tmp["subtype"] = subtype
        tmp["unit_class"] = u
        tmp["unit"] = zo_units.loc[u, "tech_type"]
        # tmp["validity_range_units"] = "Mgallons/day"
        # tmp["validity_range_lb"] = 1
        # tmp["validity_range_ub"] = 25
        ion_ex_res = pd.concat([ion_ex_res, tmp])
    return ion_ex_res


ion_ex_res = run_ion_exchange_sweep()
ion_ex_res.to_csv("results/ion_exchange_flow_tds_sweep_results.csv", index=False)

# RUN UV SWEEP

In [ ]:
def run_uv_sweep():
    u = "UVZO"
    comps = ["tds"]
    subtype = "default"
    flow_mgd = 1
    conc = 100
    n_flow_pts = 20
    n_tds_pts = 20
    # Below I am just checking our SEC values using validation data from the EPA UV Disinfection Guidance Manual (LT2ESWTR) from Nov. 2006,
    # which provides SEC values for various real UV reactor configurations for different lamp types in Appendix F.

    # SMELL TEST 1 - LPHO lamps
    # Table F.10 UV Reactor Configuration for LPHO lamps
    # Also costing data is provided for this example,
    # $2230000 (2006 dollars) total, broken down as:
    # - $1210000 for UV equipment
    # - $400000 for UV building
    # - $250000 for ancillary piping, valves, controls
    # - $360000 for electrical improvements
    q_per_unit = 15.3 * pyunits.Mgallons / pyunits.day
    n_units = 3  # 3 duty, 1 standby
    n_banks_per_unit = 6
    n_lamps_per_bank = 12
    lamp_power = 360 * pyunits.watt
    total_power = n_units * n_banks_per_unit * n_lamps_per_bank * lamp_power
    total_flow = n_units * q_per_unit
    specific_energy = total_power / total_flow
    specific_energy = pyunits.convert(
        specific_energy, to_units=pyunits.kWh / pyunits.m**3
    )
    print(f"LPHO SEC 1, check from EPA: {value(specific_energy):.5f}")

    # SMELL TEST 2 - LPHO lamps
    # Table F.15 UV Reactor Configuration for LPHO lamps
    # Also costing data provided for this example, $2170000 (2006 dollars) total
    q_per_unit = 6 * pyunits.Mgallons / pyunits.day
    n_units = 2  # 2 duty, 1 standby
    n_lamps_per_unit = 30
    lamp_power = 275 * pyunits.watt
    total_power = n_units * n_lamps_per_unit * lamp_power
    total_flow = n_units * q_per_unit
    specific_energy = total_power / total_flow
    specific_energy = pyunits.convert(
        specific_energy, to_units=pyunits.kWh / pyunits.m**3
    )
    print(f"LPHO SEC 2, check from EPA: {value(specific_energy):.5f}")

    # SMELL TEST 3 - MP lamps
    # Table F.23 UV Reactor Configuration for MP lamps
    # Also costing data provided for this example, $2170000 (2006 dollars) total
    q_per_unit = 22 * pyunits.Mgallons / pyunits.day
    n_units = 6  # 6 duty, 0 standby
    n_lamps_per_unit = 9
    lamp_power = 21.6 * pyunits.kW
    total_power = n_units * n_lamps_per_unit * lamp_power
    total_flow = n_units * q_per_unit
    specific_energy = total_power / total_flow
    specific_energy = pyunits.convert(
        specific_energy, to_units=pyunits.kWh / pyunits.m**3
    )
    print(f"MP SEC 1, check from EPA: {value(specific_energy):.5f}")

    # SMELL TEST 4 - MP lamps
    # Table F.3 UV Equipment Configuration for MP lamps
    q_per_unit = 10 * pyunits.Mgallons / pyunits.day
    n_units = 4  # 4 duty, 0 standby
    n_lamps_per_unit = 8
    lamp_power = 10 * pyunits.kW
    total_power = n_units * n_lamps_per_unit * lamp_power
    total_flow = n_units * q_per_unit
    specific_energy = total_power / total_flow
    specific_energy = pyunits.convert(
        specific_energy, to_units=pyunits.kWh / pyunits.m**3
    )
    print(f"MP SEC 2, check from EPA: {value(specific_energy):.5f}")
    # default for uv_zo is 0.1 kWh/m3, which is a bit high based on these 4 examples

    flow_sweep = np.linspace(1, 25, n_flow_pts)
    sec_sweep = [0.01, 0.025, 0.05, 0.1]  # kWh/m3, based on the above examples

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.energy_electric_flow_vol_inlet.fix(0.01)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    uv_res = pd.DataFrame()
    rd = build_results_dict(m)
    rd["flow_mgd"] = list()
    rd["sec"] = list()
    for flow in flow_sweep:
        for sec in sec_sweep:
            print(f"\nRunning UV with flow = {flow} MGD, SEC = {sec} kWh/m3\n\n")
            m = build_zo_model(
                u, comps=comps, subtype=subtype, flow_mgd=flow, conc=conc
            )
            m.fs.unit.energy_electric_flow_vol_inlet.fix(sec)

            m.fs.unit.initialize()

            results = solver.solve(m, tee=False)
            assert_optimal_termination(results)
            rd = results_dict_append(m, rd)
            rd["flow_mgd"].append(flow)
            rd["sec"].append(sec)
            print(f"LCOW = {value(m.fs.costing.LCOW):.4f} $/m3")
            print(f"SEC = {value(m.fs.costing.SEC):.4f} kWh/m3")

    uv_res = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
    uv_res["subtype"] = subtype
    uv_res["unit_class"] = u
    uv_res["unit"] = zo_units.loc[u, "tech_type"]

    return uv_res


uv_res = run_uv_sweep()
uv_res.to_csv("results/uv_flow_sec_sweep_results.csv", index=False)

# RUN UV+AOP SWEEP

In [ ]:
def run_uv_aop_sweep():
    u = "UVAOPZO"
    comps = ["tds"]
    subtype = "default"
    flow_mgd = 1
    conc = 100
    n_flow_pts = 20
    n_tds_pts = 20

    flow_sweep = np.linspace(1, 25, n_flow_pts)
    sec_sweep = [0.01, 0.025, 0.05, 0.1]  # kWh/m3, based on the UV examples above
    oxidant_dose_sweep = [1, 10]  # mg/L

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.energy_electric_flow_vol_inlet.fix(0.01)
    m.fs.unit.oxidant_dose.fix(1 * pyunits.mg / pyunits.L)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    uv_aop_res = pd.DataFrame()
    rd = build_results_dict(m)
    rd["flow_mgd"] = list()
    rd["sec"] = list()
    rd["oxidant_dose"] = list()
    for flow in flow_sweep:
        for sec in sec_sweep:
            for oxidant_dose in oxidant_dose_sweep:
                print(
                    f"\nRunning UV AOP with flow = {flow} MGD, SEC = {sec} kWh/m3, oxidant dose = {oxidant_dose} mg/L\n\n"
                )
                m = build_zo_model(
                    u, comps=comps, subtype=subtype, flow_mgd=flow, conc=conc
                )
                m.fs.unit.energy_electric_flow_vol_inlet.fix(sec)
                m.fs.unit.oxidant_dose.fix(oxidant_dose * pyunits.mg / pyunits.L)

                m.fs.unit.initialize()

                results = solver.solve(m, tee=False)
                assert_optimal_termination(results)
                rd = results_dict_append(m, rd)
                rd["flow_mgd"].append(flow)
                rd["sec"].append(sec)
                rd["oxidant_dose"].append(oxidant_dose)
                print(f"LCOW = {value(m.fs.costing.LCOW):.4f} $/m3")
                print(f"SEC = {value(m.fs.costing.SEC):.4f} k   Wh/m3")
    uv_aop_res = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
    uv_aop_res["subtype"] = subtype
    uv_aop_res["unit_class"] = u
    uv_aop_res["unit"] = zo_units.loc[u, "tech_type"]

    return uv_aop_res


uv_aop_res = run_uv_aop_sweep()
uv_aop_res.to_csv("results/uv_aop_flow_sec_oxidant_sweep_results.csv", index=False)

# RUN EVAPORATION POND SWEEP

In [ ]:
def run_evaporation_pond_sweep():
    u = "EvaporationPondZO"
    comps = ["tds"]
    subtype = "default"
    flow_mgd = 1
    conc = 130000
    n_flow_pts = 50
    n_rad_pts = 5

    flow_sweep = np.linspace(0.05, 0.5, n_flow_pts)  # brine flow
    flow_sweep = np.linspace(0.05, 1, n_flow_pts)  # brine flow
    rad_sweep = np.linspace(
        15, 25, n_rad_pts
    )  # solar radiation in MJ/m2/day, NREL for North America

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    evap_pond_res = pd.DataFrame()
    rd = build_results_dict(m)
    rd["flow_mgd"] = list()
    rd["solar_radiation"] = list()
    for flow in flow_sweep:
        for rad in rad_sweep:
            # for tds in tds_sweep:
            print(
                f"\nRunning Evaporation Pond with flow = {flow} MGD, solar radiation = {rad} W/m2\n\n"
            )
            m = build_zo_model(
                u, comps=comps, subtype=subtype, flow_mgd=flow, conc=conc
            )
            m.fs.unit.solar_radiation.fix(rad)

            m.fs.unit.initialize()

            results = solver.solve(m, tee=False)
            assert_optimal_termination(results)
            rd = results_dict_append(m, rd)
            rd["flow_mgd"].append(flow)
            rd["solar_radiation"].append(rad)
            print(f"area = {value(m.fs.unit.area):.2f} acres")
            print(f"adj-area = {value(m.fs.unit.adj_area):.2f} acres")
            print(f"LCOW = {value(m.fs.costing.LCOW):.4f} $/m3")
            print(f"CAPEX = {value(m.fs.costing.total_capital_cost):.2f} $")
            clear_output(wait=True)

        # rd["tds"].append(tds)

    evap_pond_res = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
    evap_pond_res["subtype"] = subtype
    evap_pond_res["unit_class"] = u
    evap_pond_res["unit"] = zo_units.loc[u, "tech_type"]
    return evap_pond_res


evap_pond_res = run_evaporation_pond_sweep()
evap_pond_res.to_csv("results/evaporation_pond_flow_sweep_results.csv", index=False)


# RUN BRINE CONCENTRATOR SWEEP

In [ ]:
def run_brine_concentrator_sweep():
    u = "BrineConcentratorZO"
    comps = ["tds"]
    subtype = "default"
    flow_mgd = 1
    conc = 130000
    n_flow_pts = 50
    n_tds_pts = 5

    # flow_sweep = np.linspace(0.06, 0.5, n_flow_pts) # brine flow
    flow_sweep = np.linspace(0.06, 20, n_flow_pts)  # brine flow, this is validity range
    flow_sweep = np.linspace(0.05, 1, n_flow_pts)  # brine flow
    tds_sweep = np.linspace(10000, 60000, n_tds_pts)  # TDS in mg/L
    recovery_sweep = [0.9, 0.95, 0.99]
    recov = 0.96

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    brine_conc_res = pd.DataFrame()
    rd = build_results_dict(m)
    rd["flow_mgd"] = list()
    rd["tds"] = list()
    for flow in flow_sweep:
        for tds in tds_sweep:
            print(
                f"\nRunning Brine Concentrator with flow = {flow} MGD, TDS = {tds} mg/L, Recovery = {recov}\n\n"
            )
            m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow, conc=tds)
            m.fs.unit.recovery_frac_mass_H2O.fix(recov)
            m.fs.unit.initialize()

            results = solver.solve(m, tee=False)
            assert_optimal_termination(results)
            rd = results_dict_append(m, rd)
            rd["flow_mgd"].append(flow)
            rd["tds"].append(tds)
            print(f"LCOW = {value(m.fs.costing.LCOW):.4f} $/m3")
            print(f"SEC = {value(m.fs.costing.SEC):.4f} kWh/m3")
            print(f"CAPEX = {value(m.fs.costing.total_capital_cost):.2f} $")
            clear_output(wait=True)

        # rd["tds"].append(tds)

    brine_conc_res = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
    brine_conc_res["subtype"] = subtype
    brine_conc_res["unit_class"] = u
    brine_conc_res["unit"] = zo_units.loc[u, "tech_type"]
    return brine_conc_res


brine_conc_res = run_brine_concentrator_sweep()
brine_conc_res.to_csv("results/brine_concentrator_flow_tds_sweep_results.csv", index=False)

# RUN CRYSTALLIZER SWEEP

In [ ]:
def run_crystallizer_sweep():
    u = "CrystallizerZO"
    comps = ["tds"]
    subtype = "default"
    flow_mgd = 1
    conc = 130000
    flow_lb = 11 * pyunits.gallon / pyunits.minute
    flow_ub = 1047 * pyunits.gallon / pyunits.minute
    flow_lb_mgd = pyunits.convert(flow_lb, to_units=pyunits.Mgallons / pyunits.day)()
    flow_ub_mgd = pyunits.convert(flow_ub, to_units=pyunits.Mgallons / pyunits.day)()
    n_flow_pts = 50
    n_tds_pts = 5
    recov = 0.99

    # flow_sweep = np.linspace(flow_lb_mgd, flow_ub_mgd, n_flow_pts)  # brine flow
    flow_sweep = np.linspace(0.05, 1, n_flow_pts)  # brine flow
    tds_sweep = np.linspace(145000, 358000, n_tds_pts)  # TDS in mg/L

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    crystallizer_res = pd.DataFrame()
    rd = build_results_dict(m)
    rd["flow_mgd"] = list()
    rd["tds"] = list()
    for flow in flow_sweep:
        for tds in tds_sweep:
            print(
                f"\nRunning Crystallizer with flow = {flow} MGD, TDS = {tds} mg/L\n\n"
            )
            m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow, conc=tds)
            m.fs.unit.recovery_frac_mass_H2O.fix(recov)

            m.fs.unit.initialize()

            results = solver.solve(m, tee=False)
            assert_optimal_termination(results)
            rd = results_dict_append(m, rd)
            rd["flow_mgd"].append(flow)
            rd["tds"].append(tds)
            print(f"LCOW = {value(m.fs.costing.LCOW):.4f} $/m3")
            print(f"SEC = {value(m.fs.costing.SEC):.4f} kWh/m3")
            print(f"CAPEX = {value(m.fs.costing.total_capital_cost):.2f} $")
            clear_output(wait=True)

        # rd["tds"].append(tds)

    crystallizer_res = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
    crystallizer_res["subtype"] = subtype
    crystallizer_res["unit_class"] = u
    crystallizer_res["unit"] = "crystallizer"
    return crystallizer_res


crystallizer_res = run_crystallizer_sweep()
crystallizer_res.to_csv("results/crystallizer_flow_tds_sweep_results.csv", index=False)

# RUN FILTER PRESS SWEEP

In [ ]:
def run_filter_press_sweep():
    u = "FilterPressZO"
    comps = ["tss"]
    subtype = "default"
    flow_mgd = 1
    conc = 100
    n_flow_pts = 20
    n_tds_pts = 20

    flow_sweep = np.linspace(0.05, 0.5, n_flow_pts)  # brine flow
    hrs_per_day_sweep = [3, 6, 12, 24]
    # flow_sweep = np.linspace(1, 25, n_flow_pts)
    # tds_sweep = np.linspace(1000, 100000, n_tds_pts)

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    filter_press_res = pd.DataFrame()
    rd = build_results_dict(m)
    rd["flow_mgd"] = list()
    rd["hrs_per_day"] = list()
    for flow in flow_sweep:
        for hrs_per_day in hrs_per_day_sweep:
            print(
                f"\nRunning Filter Press with flow = {flow} MGD, TSS = {conc} mg/L, Hours per day = {hrs_per_day}\n\n"
            )
            m = build_zo_model(
                u, comps=comps, subtype=subtype, flow_mgd=flow, conc=conc
            )
            m.fs.unit.hours_per_day_operation.fix(
                hrs_per_day * pyunits.hours / pyunits.day
            )
            m.fs.unit.initialize()

            results = solver.solve(m, tee=False)
            assert_optimal_termination(results)
            rd = results_dict_append(m, rd)
            rd["flow_mgd"].append(flow)
            rd["hrs_per_day"].append(hrs_per_day)
            print(f"LCOW = {value(m.fs.costing.LCOW):.4f} $/m3")
            print(f"SEC = {value(m.fs.costing.SEC):.4f} kWh/m3")
            print(f"CAPEX = {value(m.fs.costing.total_capital_cost):.2f} $")
        clear_output(wait=True)

        # rd["tds"].append(tds)

    filter_press_res = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
    filter_press_res["subtype"] = subtype
    filter_press_res["unit_class"] = u
    filter_press_res["unit"] = zo_units.loc[u, "tech_type"]
    return filter_press_res


filter_press_res = run_filter_press_sweep()
filter_press_res.to_csv("results/filter_press_flow_sweep_results.csv", index=False)

# RUN ELECTROCOAGULATION SWEEP

In [ ]:
def run_electocoagulation_sweep():
    u = "ElectrocoagulationZO"
    comps = ["tds"]
    subtype = "default"
    flow_mgd = 1
    conc = 100
    n_flow_pts = 20
    n_tds_pts = 10

    flow_sweep = np.linspace(1, 25, n_flow_pts)  # brine flow
    # flow_sweep = np.linspace(1, 25, n_flow_pts)
    tds_sweep = np.linspace(1000, 10000, n_tds_pts)
    dose_sweep = [1, 10, 100]  # mg/L, just checking sensitivity to this parameter

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    ec_res = pd.DataFrame()
    rd = build_results_dict(m)
    rd["flow_mgd"] = list()
    rd["tds"] = list()
    rd["metal_dose"] = list()
    rd["elec_material"] = list()
    for flow in flow_sweep:
        for tds in tds_sweep:
            for metal_dose in dose_sweep:
                for elec_material in ["aluminum", "iron"]:
                    c = {"electrode_material": elec_material}
                    print(
                        f"\nRunning {elec_material.upper()} Electrocoagulation with flow = {flow} MGD, TDS = {tds} mg/L, dose = {metal_dose} mg/L\n\n"
                    )
                    try:
                        m = build_zo_model(
                            u,
                            comps=comps,
                            subtype=subtype,
                            flow_mgd=flow,
                            conc=tds,
                            unit_config=c,
                        )

                        m.fs.unit.metal_dose.fix(metal_dose * pyunits.mg / pyunits.L)
                        m.fs.costing.electrocoagulation.ec_reactor_cap_material_coeff.fix(
                            1
                        )
                        m.fs.unit.initialize()

                        results = solver.solve(m, tee=False)
                        assert_optimal_termination(results)
                        rd = results_dict_append(m, rd)
                        rd["flow_mgd"].append(flow)
                        rd["tds"].append(tds)
                        rd["metal_dose"].append(metal_dose)
                        rd["elec_material"].append(elec_material)
                        print(f"LCOW = {value(m.fs.costing.LCOW):.4f} $/m3")
                        print(f"SEC = {value(m.fs.costing.SEC):.4f} kWh/m3")
                        print(f"CAPEX = {value(m.fs.costing.total_capital_cost):.2f} $")
                    except:
                        print(
                            f"Model failed to solve for {elec_material.upper()} Electrocoagulation with flow = {flow} MGD, TDS = {tds} mg/L, dose = {metal_dose} mg/L"
                        )
                        continue
            clear_output(wait=True)

    tmp = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
    tmp["subtype"] = subtype
    tmp["unit_class"] = u
    tmp["unit"] = zo_units.loc[u, "tech_type"]
    ec_res = pd.concat([ec_res, tmp])
    return ec_res


electrocoag_res = run_electocoagulation_sweep()
electrocoag_res.to_csv("results/electrocoagulation_flow_tds_sweep_results.csv", index=False)

# RUN COAG AND FLOC

In [ ]:
def run_coag_and_floc_sweep():
    u = "CoagulationFlocculationZO"
    comps = ["tss"]
    subtype = "default"
    flow_mgd = 1
    conc = 100
    n_flow_pts = 50
    n_tds_pts = 10

    flow_sweep = np.linspace(1, 25, n_flow_pts) 
    dose_sweep = [1, 10, 100]  

    m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow_mgd, conc=conc)
    m.fs.unit.initialize()
    m.fs.costing.initialize()

    results = solver.solve(m, tee=False)
    assert_optimal_termination(results)

    coag_floc_res = pd.DataFrame()
    rd = build_results_dict(m)
    rd["flow_mgd"] = list()
    # rd["tds"] = list()
    # rd["coagulant_dose"] = list()
    for flow in flow_sweep:
        # for tds in tds_sweep:
        #     for coagulant_dose in dose_sweep:
        print(f"\nRunning Coagulation and Flocculation with flow = {flow} MGD\n\n")
        m = build_zo_model(u, comps=comps, subtype=subtype, flow_mgd=flow, conc=conc)

        m.fs.unit.initialize()

        results = solver.solve(m, tee=False)
        assert_optimal_termination(results)
        rd = results_dict_append(m, rd)
        rd["flow_mgd"].append(flow)
        print(f"LCOW = {value(m.fs.costing.LCOW):.4f} $/m3")
        print(f"SEC = {value(m.fs.costing.SEC):.4f} kWh/m3")
        print(f"CAPEX = {value(m.fs.costing.total_capital_cost):.2f} $")
    coag_floc_res = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
    coag_floc_res["subtype"] = subtype
    coag_floc_res["unit_class"] = u
    coag_floc_res["unit"] = zo_units.loc[u, "tech_type"]
    return coag_floc_res


coag_floc_res = run_coag_and_floc_sweep()
coag_floc_res.to_csv("results/coag_and_floc_flow_sweep_results.csv", index=False)

# RUNNING RO 1D SWEEP

need to have multistage RO fs for this to run

In [ ]:
# import watertap.kurby.development.multistage_RO.multistage_RO as msro
# pump_dict = {1: True, 2: False, 3: True, 4: False, 5: True}
# flow_mgd = 1 * pyunits.Mgallons / pyunits.day
# flow_vol = value(pyunits.convert(flow_mgd, to_units=pyunits.liter / pyunits.s))
# salinity = 50
# add_erd = False
# m = msro.run_n_stage_system(
#     n_stages=2,
#     salinity=salinity,
#     flow_vol=1,
#     add_erd=add_erd,
#     pump_dict=pump_dict,
#     max_pressure=85e5,
# )

# m = msro.set_system_recovery(m, 0.5)

# res = msro.utils.solve(model=m, tee=False)
# # rd = build_results_dict(m)
# # rd["flow_mgd"] = list()
# # rd["tds"] = list()
# # salinity = [35, 70]
# # flows = np.linspace(0.1, 20, 20)
# # for flow_mgd in flows:
# #     flow_vol = value(
# #         pyunits.convert(
# #             flow_mgd * pyunits.Mgallons / pyunits.day,
# #             to_units=pyunits.liter / pyunits.s,
# #         )
# #     )
# #     m = msro.run_n_stage_system(
# #         n_stages=1,
# #         salinity=salinity,
# #         flow_vol=flow_vol,
# #         add_erd=add_erd,
# #         pump_dict=pump_dict,
# #         # max_pressure=300e5
# #     )

# #     m = msro.set_system_recovery(m, 0.5)

# #     res = msro.utils.solve(model=m, tee=False)

# #     assert_optimal_termination(res)
# #     rd = results_dict_append(m, rd)
# #     rd["flow_mgd"].append(flow_mgd)

# # results = pd.DataFrame({k: pd.Series(v) for k, v in rd.items()})
# # results.to_csv("results/ro_sensitivity_flow.csv", index=False)

# msro.utils.report_n_stage_system(m)